# 06. Pipeline Parallelism GPipe 1F1B Zero Bubble and DualPipe

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Huang et al. (GPipe, 2019, *GPipe: Efficient Training of Giant Neural Networks Using Pipeline Parallelism*), Narayanan et al. (PipeDream, 2019), Qi et al. (Zero-Bubble PP, 2024), DeepSeek-AI (DualPipe, 2024, *DeepSeek-V3 Technical Report*).
- **Purpose:** Partition layers sequentially across stages and schedule microbatches to eliminate idle pipeline bubbles while bounding activation memory.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Context Parallelism Ring Attention and Ulysses](05_context_parallelism_ring_attention_and_ulysses.ipynb) $\to$ `🔗 Downstream Syntheses:` [Expert Parallelism and 3D 4D Parallel Composition](07_expert_parallelism_and_3d_4d_parallel_composition.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use Pipeline Parallelism when the number of microbatches $M$ is small relative to pipeline stages $P$ ($M < 4P$). The pipeline bubble $F_{\text{bubble}} = \frac{P-1}{M+P-1}$ dominates runtime unless $M \gg P$, or when low inter-layer tensor sizes enable standard Data/ZeRO Parallelism.

## Mathematical Foundations & Governing Equations

$$
F_{\text{bubble}} = \frac{P - 1}{M + P - 1}, \quad B_A = \delta W^\top, \quad B_W = X^\top \delta, \quad M_{\text{1F1B}} = O(P) \ll O(M) = M_{\text{GPipe}}
$$

### Asymptotic Time & Memory Complexity
- **Bubble Reduction:** Zero-Bubble and DualPipe decouple $B_A$ (activation grad, latency-critical) and $B_W$ (weight grad, deferred), hiding $B_W$ inside bubble slots.
- **Memory Footprint:** 1F1B caps active microbatch buffers to $P$ stages, avoiding the $O(M)$ memory accumulation of GPipe.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def compute_pipeline_bubble_fraction(num_stages: int, num_microbatches: int) -> float:
    return (num_stages - 1) / (num_microbatches + num_stages - 1)

def pipeline_forward_step(x_mb: torch.Tensor, w0: torch.Tensor, b0: torch.Tensor, w1: torch.Tensor, b1: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    m = x_mb.shape[0]
    h_list = [x_mb[i] @ w0 + b0 for i in range(m)]
    y_list = [h_list[i] @ w1 + b1 for i in range(m)]
    return torch.stack(h_list, dim=0), torch.stack(y_list, dim=0)

def dualpipe_backward_step(x_mb: torch.Tensor, h_mb: torch.Tensor, grad_y_mb: torch.Tensor, w0: torch.Tensor, w1: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    m = x_mb.shape[0]
    grad_w0, grad_w1 = torch.zeros_like(w0), torch.zeros_like(w1)
    grad_x_list = []
    for i in range(m):
        b_a_1 = grad_y_mb[i] @ w1.t()
        grad_w1 += h_mb[i].t() @ grad_y_mb[i]
        b_a_0 = b_a_1 @ w0.t()
        grad_w0 += x_mb[i].t() @ b_a_1
        grad_x_list.append(b_a_0)
    return grad_w0, grad_w1, torch.stack(grad_x_list, dim=0)

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

bubble_f = compute_pipeline_bubble_fraction(4, 4)
assert abs(bubble_f - 3.0 / 7.0) < 1e-6

h_mb, y_mb = pipeline_forward_step(weights["x_mb"], weights["w0"], weights["b0"], weights["w1"], weights["b1"])
assert float(torch.max(torch.abs(h_mb - weights["stage0_fwd_outs"]))) < 1e-5
assert float(torch.max(torch.abs(y_mb - weights["stage1_fwd_outs"]))) < 1e-5

grad_w0, grad_w1, grad_x = dualpipe_backward_step(weights["x_mb"], weights["stage0_fwd_outs"], weights["grad_y_mb"], weights["w0"], weights["w1"])
assert float(torch.max(torch.abs(grad_w0 - weights["grad_w0"]))) < 1e-5
assert float(torch.max(torch.abs(grad_w1 - weights["grad_w1"]))) < 1e-5
assert float(torch.max(torch.abs(grad_x - weights["grad_x_outs"]))) < 1e-5
print("All Pipeline Parallelism (GPipe, 1F1B, DualPipe) parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: pipeline_forward_step(weights["x_mb"], weights["w0"], weights["b0"], weights["w1"], weights["b1"]), iters=50, name="Pipeline Forward Step"),
    benchmark_fn(lambda: dualpipe_backward_step(weights["x_mb"], weights["stage0_fwd_outs"], weights["grad_y_mb"], weights["w0"], weights["w1"]), iters=50, name="DualPipe Backward Step"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: GPipe O(M) Activation Buffer Explosion vs 1F1B O(P) Bounding
num_stages = 8
num_microbatches = 64
act_per_microbatch_mb = 128.0  # 128 MB per microbatch

# GPipe holds activations for all M microbatches on stage 0 before backward returns:
gpipe_peak_mb = num_microbatches * act_per_microbatch_mb

# 1F1B alternates forward/backward, bounding active microbatches to P stages:
one_f_one_b_peak_mb = num_stages * act_per_microbatch_mb

print(f"GPipe Peak Activation Memory: {gpipe_peak_mb / 1024:.2f} GB / GPU")
print(f"1F1B Peak Activation Memory:  {one_f_one_b_peak_mb / 1024:.2f} GB / GPU  ({gpipe_peak_mb / one_f_one_b_peak_mb:.1f}x reduction)")
assert gpipe_peak_mb == (num_microbatches / num_stages) * one_f_one_b_peak_mb, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Inter-Layer Sharding Efficiency:** Pipeline Parallelism requires communication only across boundary layers, scaling across low-bandwidth inter-node fabrics.
2. **1F1B Memory Bounding:** Interleaving forward and backward passes caps peak activation memory at $O(P)$ rather than $O(M)$.
3. **Zero-Bubble Decoupling:** Separating latency-critical activation gradients ($B_A$) from latency-tolerant weight gradients ($B_W$) reclaims idle bubble cycles.

---
### Cross-Domain Generalisation Matrix
| Domain | Staged Pipeline Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Instruction-Level Superscalar CPUs** | Classic 5-stage RISC instruction pipeline | Sustains high throughput via overlapping instruction fetch/decode/execute/writeback |
| **Automated Manufacturing Assembly** | Station assembly line with buffer queues | Eliminates idle station stalls by decoupling sub-component delivery |
| **Video Streaming Transcoding** | Staged demux $\to$ decode $\to$ filter $\to$ encode pipeline | Bounds intermediate frame queue depth using 1F1B backpressure |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.md](../../markdown/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.md)
- **PyTorch 2.x Module:** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.py](../../pytorch2/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.py)
- **Functional JAX Module:** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.py](../../jax/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.py)
- **Native Rust Candle Track:** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.rs](../../candle/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.rs)
- **Hardware Kernel (CPU / WGSL):** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.rs](../../kernels/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.rs)
- **Golden SafeTensors:** [06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.safetensors](../../weights/16_distributed_and_scalable/06_pipeline_parallelism_gpipe_1f1b_zero_bubble_and_dualpipe.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Context Parallelism Ring Attention and Ulysses](05_context_parallelism_ring_attention_and_ulysses.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 07. Expert Parallelism and 3D 4D Parallel Composition](07_expert_parallelism_and_3d_4d_parallel_composition.ipynb) &rarr;